In [ ]:
import psycopg2
from kafka import KafkaConsumer
import json

In [ ]:
import psycopg2
from kafka import KafkaConsumer
import json

conn = psycopg2.connect(
    host="localhost",
    database="reddit_db",
    user="postgres",
    password="25524",
    port="5432" 
)

cursor = conn.cursor()

consumer = KafkaConsumer(
    'comments_sentiment',
    bootstrap_servers=['127.0.0.1:9092'],
    auto_offset_reset='earliest',
    enable_auto_commit=True,
    group_id='sentiment_group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Consumer Kafka connecté")


for message in consumer:

    data = message.value

    comments = data.get('comments')
    sentiment = data.get('sentiment')

    cursor.execute(
        "INSERT INTO comments (comments, sentiment) VALUES (%s, %s)",
        (comments, sentiment)
    )
    conn.commit()

    print(f"Comments inséré : {comments}... | sentiment : {sentiment}")